# 🎬 Pipeline Multilíngue — Orações com Legendas Morfológicas

**Estratégia de classificação intermediária:**
O Groq gera as classificações → você revisa com uma IA → pipeline continua com os JSONs corrigidos.

---

### Fluxo completo

| # | Fase | O que faz | Ação |
|---|------|-----------|------|
| 0 | Setup | Instala deps, monta Drive, importa módulos | Automático |
| Init | — | Cria config, groq e pipeline | Automático |
| 1 | Áudio | Edge TTS → .wav no Drive | Automático |
| 2 | Whisper | Transcrição → SRT bruto | Automático |
| 3 | Correção PT | Groq corrige o SRT | Automático |
| 4 | Traduções | Groq gera EN/ES/FR | Automático |
| **5A** | **Classificação** | **Groq classifica → pausa** | **Automático → pausa** |
| **5B** | **Revisão** | **Você corrige JSONs com IA** | **👤 Manual** |
| **5C** | **Recarregar** | **Pipeline carrega JSONs corrigidos** | **Automático** |
| 6 | Clipes | Baixa e corta vídeos do Pixabay | Automático |
| 7 | Vídeo base | Crédito + narração + trilha | Automático |
| 8 | Legendas ASS | Queima legendas coloridas no vídeo | Automático |

> **Retomar de uma fase:** `pipeline.run(from_phase='nome_da_fase')`


In [13]:
# ╔══════════════════════════════════════════════════╗
# ║  CÉLULA 0 — Setup (rode uma vez por sessão)    ║
# ╚══════════════════════════════════════════════════╝

# Sistema
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# Python
!pip install -q edge-tts openai-whisper openai pandas gdown yt-dlp
print('✅ pacotes Python')

# Drive
from google.colab import drive, userdata
drive.mount('/content/drive')
print('✅ Drive montado')

# Copiar módulos do Drive para /content/pipeline
import shutil, os, sys, logging

PASTA_MODULOS = '/content/drive/MyDrive/pipeline/modulos'  # <-- ajuste se necessário
DESTINO       = '/content/pipeline'

if os.path.exists(PASTA_MODULOS):
    shutil.copytree(PASTA_MODULOS, DESTINO, dirs_exist_ok=True)
    print(f'✅ Módulos copiados: {PASTA_MODULOS} → {DESTINO}')
else:
    print(f'⚠️  Pasta não encontrada: {PASTA_MODULOS}')
    print('   Ajuste PASTA_MODULOS acima.')

if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(name)-22s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)

os.chdir('/content')
print('✅ Pronto.')

✅ ffmpeg + espeak-ng
✅ pacotes Python
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado
✅ Módulos copiados: /content/drive/MyDrive/pipeline/modulos → /content/pipeline
✅ Pronto.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 VERIFICAÇÃO DA ESTRUTURA DO DRIVE                           ║
# ║  (execute após o SETUP, antes da INICIALIZAÇÃO)                 ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 60)
print("📁 VERIFICANDO ESTRUTURA DO DRIVE")
print("═" * 60)

# Pasta principal de correcoes
correcoes = Path('/content/drive/MyDrive/pipeline/correcoes')

if not correcoes.exists():
    print("⚠️ Pasta 'correcoes' não encontrada!")
    print("   Criando estrutura...")
    correcoes.mkdir(parents=True, exist_ok=True)

# Verificar arquivos genéricos
print("\n📄 ARQUIVOS GENÉRICOS (devem estar na pasta correcoes/):")
for arquivo in ['prompt_revisao.md', 'relatorio_classificacoes.csv']:
    caminho = correcoes / arquivo
    if caminho.exists():
        tamanho = caminho.stat().st_size / 1024
        print(f"   ✅ {arquivo} ({tamanho:.1f} KB)")
    else:
        print(f"   ❌ {arquivo} - NÃO ENCONTRADO")
        print(f"      💡 Você precisa criar este arquivo manualmente")

# Verificar pasta da oração atual (se já existir)
print(f"\n📁 PASTA DA ORAÇÃO ATUAL: {config.NOME_ORACAO if 'config' in dir() else '(inicializar primeiro)'}")

if 'config' in dir():
    pasta_oracao = correcoes / config.NOME_ORACAO
    if pasta_oracao.exists():
        print(f"   ✅ Pasta encontrada")
        for lang in ['pt', 'en', 'es', 'fr']:
            json_path = pasta_oracao / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
            if json_path.exists():
                tamanho = json_path.stat().st_size / 1024
                print(f"      ✅ {json_path.name} ({tamanho:.1f} KB)")
            else:
                print(f"      ❌ {json_path.name}")
    else:
        print(f"   ⚠️ Pasta ainda não criada (será criada na FASE 5A)")
else:
    print("   ⚠️ Inicialize o pipeline primeiro para verificar a pasta da oração")

print("\n" + "═" * 60)
print("💡 ESTRUTURA ESPERADA:")
print("   MyDrive/pipeline/correcoes/")
print("   ├── prompt_revisao.md")
print("   ├── relatorio_classificacoes.csv")
print("   └── pai_nosso/")
print("       ├── classificacao_pai_nosso_pt.json")
print("       ├── classificacao_pai_nosso_en.json")
print("       ├── classificacao_pai_nosso_es.json")
print("       └── classificacao_pai_nosso_fr.json")
print("═" * 60)

In [12]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 LIMPEZA SELETIVA DO PIPELINE                                ║
# ║  Marque o que quer limpar e clique no botão                     ║
# ║  (Coloque esta célula após o SETUP, antes da INICIALIZAÇÃO)    ║
# ╚══════════════════════════════════════════════════════════════════╝

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pathlib import Path
import shutil
import sys
import os

output_limpeza = widgets.Output()

def limpeza_rapida(tipo):
    with output_limpeza:
        clear_output(wait=True)

        print("═" * 65)
        print(f"🧹 LIMPANDO: {tipo}")
        print("═" * 65)

        cont = 0

        # 1. 🎵 Áudios
        if tipo == 'audio' or tipo == 'tudo':
            print("\n🎵 Removendo áudios...")
            for f in Path('.').glob('*_audio.wav'):
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")
            for f in Path('.').glob('*.mp3'):
                if 'musica' not in f.name:
                    f.unlink()
                    cont += 1
                    print(f"   🗑️ {f.name}")

        # 2. 📝 Legendas SRT e ASS
        if tipo == 'legendas' or tipo == 'tudo':
            print("\n📝 Removendo legendas...")
            for f in Path('.').glob('*.srt'):
                if 'yt_ref' not in f.name:
                    f.unlink()
                    cont += 1
                    print(f"   🗑️ {f.name}")
            for f in Path('.').glob('*.ass'):
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

        # 3. 🎬 Vídeos
        if tipo == 'videos' or tipo == 'tudo':
            print("\n🎬 Removendo vídeos...")
            patterns = ['*_base.mp4', '*_final.mp4', 'clipe_*.mp4', 'temp_*.mp4', 'video_*.mp4', 'raw_*.mp4']
            for pattern in patterns:
                for f in Path('.').glob(pattern):
                    f.unlink()
                    cont += 1
                    print(f"   🗑️ {f.name}")

        # 4. 📊 JSONs de classificação (LOCAIS)
        if tipo == 'jsons' or tipo == 'tudo':
            print("\n📊 Removendo JSONs de classificação (LOCAIS)...")
            for f in Path('.').glob('classificacao_*.json'):
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

        # 5. 📌 Checkpoint
        if tipo == 'checkpoint' or tipo == 'tudo':
            print("\n📌 Removendo checkpoint...")
            cp = Path('checkpoint.json')
            if cp.exists():
                cp.unlink()
                cont += 1
                print("   🗑️ checkpoint.json")

        # 6. 📁 Pastas temporárias
        if tipo == 'pastas' or tipo == 'tudo':
            print("\n📁 Removendo pastas temporárias...")
            pastas = ['clipes_cortados', 'clipes_prontos', 'temp_raw', '__pycache__']
            for pasta in pastas:
                p = Path(pasta)
                if p.exists():
                    shutil.rmtree(p)
                    cont += 1
                    print(f"   🗑️ {pasta}/")

        # 7. 📦 Cache Python
        if tipo == 'cache' or tipo == 'tudo':
            print("\n📦 Limpando cache Python...")
            modulos = ['groq_client', 'video_pipeline', 'config', 'classification',
                       'checkpoint', 'drive_utils', 'ffmpeg_utils', 'srt_utils',
                       'models', 'constants', 'pipeline']
            for m in modulos:
                if m in sys.modules:
                    del sys.modules[m]
                    cont += 1
                    print(f"   🗑️ {m}")

        # 8. 🗑️ LIMPAR DRIVE (JSONs corrigidos) - SÓ SE ESCOLHIDO
        if tipo == 'drive' or tipo == 'tudo_drive':
            print("\n☁️ Removendo JSONs do Drive (pasta correcoes)...")
            drive_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes')
            if drive_correcoes.exists():
                for oracao_dir in drive_correcoes.iterdir():
                    if oracao_dir.is_dir():
                        for f in oracao_dir.glob('classificacao_*.json'):
                            f.unlink()
                            cont += 1
                            print(f"   🗑️ Drive: {oracao_dir.name}/{f.name}")
                print(f"\n   ⚠️ Mantidos: prompt_revisao.md e relatorio_classificacoes.csv")

        print("\n" + "═" * 65)
        print(f"✅ Limpeza concluída! {cont} item(ns) removido(s)")
        print("═" * 65)

# Criar botões
btn_audio = widgets.Button(description='🎵 Áudios', button_style='warning', layout=widgets.Layout(width='130px'))
btn_legendas = widgets.Button(description='📝 Legendas', button_style='warning', layout=widgets.Layout(width='130px'))
btn_videos = widgets.Button(description='🎬 Vídeos', button_style='warning', layout=widgets.Layout(width='130px'))
btn_jsons = widgets.Button(description='📊 JSONs (local)', button_style='warning', layout=widgets.Layout(width='140px'))
btn_checkpoint = widgets.Button(description='📌 Checkpoint', button_style='warning', layout=widgets.Layout(width='130px'))
btn_pastas = widgets.Button(description='📁 Pastas temp', button_style='warning', layout=widgets.Layout(width='130px'))
btn_cache = widgets.Button(description='📦 Cache Python', button_style='warning', layout=widgets.Layout(width='140px'))
btn_drive = widgets.Button(description='☁️ JSONs no Drive', button_style='danger', layout=widgets.Layout(width='150px'))
btn_tudo = widgets.Button(description='🔥 LIMPAR TUDO (local)', button_style='danger', layout=widgets.Layout(width='180px', height='45px'))
btn_tudo_drive = widgets.Button(description='🔥🔥 LIMPAR TUDO + DRIVE', button_style='danger', layout=widgets.Layout(width='220px', height='45px'))

# Conectar botões
btn_audio.on_click(lambda b: limpeza_rapida('audio'))
btn_legendas.on_click(lambda b: limpeza_rapida('legendas'))
btn_videos.on_click(lambda b: limpeza_rapida('videos'))
btn_jsons.on_click(lambda b: limpeza_rapida('jsons'))
btn_checkpoint.on_click(lambda b: limpeza_rapida('checkpoint'))
btn_pastas.on_click(lambda b: limpeza_rapida('pastas'))
btn_cache.on_click(lambda b: limpeza_rapida('cache'))
btn_drive.on_click(lambda b: limpeza_rapida('drive'))
btn_tudo.on_click(lambda b: limpeza_rapida('tudo'))
btn_tudo_drive.on_click(lambda b: limpeza_rapida('tudo_drive'))

# Título
print("═" * 65)
print("🎯 LIMPEZA SELETIVA - Clique no botão do que quer limpar")
print("═" * 65)

# Tabela explicativa
explicacao = HTML("""
<table style="width:100%; font-size:12px;">
 <tr><th style="text-align:left">Botão</th><th style="text-align:left">O que limpa</th></tr>
 <tr><td>🎵 Áudios</td><td>Arquivos .wav e .mp3 gerados</td></tr>
 <tr><td>📝 Legendas</td><td>Arquivos .srt e .ass (exceto yt_ref)</td></tr>
 <tr><td>🎬 Vídeos</td><td>Vídeos gerados (*_base.mp4, *_final.mp4, clipes)</td></tr>
 <tr><td>📊 JSONs (local)</td><td>JSONs de classificação na pasta /content/</td></tr>
 <tr><td>📌 Checkpoint</td><td>checkpoint.json (progresso do pipeline)</td></tr>
 <tr><td>📁 Pastas temp</td><td>Pastas temporárias (clipes_cortados, etc.)</td></tr>
 <tr><td>📦 Cache Python</td><td>Módulos carregados na memória</td></tr>
 <tr><td>☁️ JSONs no Drive</td><td>JSONs corrigidos em MyDrive/pipeline/correcoes/</td></tr>
 <tr><td>🔥 LIMPAR TUDO</td><td>Todos os itens acima (exceto Drive)</td></tr>
 <tr><td>🔥🔥 LIMPAR TUDO + DRIVE</td><td>TUDO, incluindo JSONs corrigidos no Drive</td></tr>
</table>
""")

display(explicacao)

# Organizar botões em linhas
row1 = widgets.HBox([btn_audio, btn_legendas, btn_videos, btn_jsons])
row2 = widgets.HBox([btn_checkpoint, btn_pastas, btn_cache, btn_drive])
row3 = widgets.HBox([btn_tudo, btn_tudo_drive])

display(row1, row2, row3)
display(output_limpeza)

print("\n💡 DICA: Use 'JSONs no Drive' para limpar classificações corrigidas")
print("   'LIMPAR TUDO' mantém os JSONs corrigidos no Drive")
print("   'LIMPAR TUDO + DRIVE' remove TUDO (use com cuidado!)")

═════════════════════════════════════════════════════════════════
🎯 LIMPEZA SELETIVA - Clique no botão do que quer limpar
═════════════════════════════════════════════════════════════════


Botão,O que limpa
🎵 Áudios,Arquivos .wav e .mp3 gerados
📝 Legendas,Arquivos .srt e .ass (exceto yt_ref)
🎬 Vídeos,"Vídeos gerados (*_base.mp4, *_final.mp4, clipes)"
📊 JSONs (local),JSONs de classificação na pasta /content/
📌 Checkpoint,checkpoint.json (progresso do pipeline)
📁 Pastas temp,"Pastas temporárias (clipes_cortados, etc.)"
📦 Cache Python,Módulos carregados na memória
☁️ JSONs no Drive,JSONs corrigidos em MyDrive/pipeline/correcoes/
🔥 LIMPAR TUDO,Todos os itens acima (exceto Drive)
🔥🔥 LIMPAR TUDO + DRIVE,"TUDO, incluindo JSONs corrigidos no Drive"


Output()


💡 DICA: Use 'JSONs no Drive' para limpar classificações corrigidas
   'LIMPAR TUDO' mantém os JSONs corrigidos no Drive
   'LIMPAR TUDO + DRIVE' remove TUDO (use com cuidado!)


---
### 🔵 Célula B0 — Opcional: baixar legendas do YouTube
Execute **antes** das fases 1–8 para ter traduções mais fiéis. Se pular, o Groq traduz a partir do PT.

In [15]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CÉLULA B0 — Baixar legendas do YouTube                        ║
# ║  (BAIXA TAMBÉM A REFERÊNCIA PT para a FASE 3)                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import subprocess
import shutil
from drive_utils import DriveClient
from config import PipelineConfig

cfg = PipelineConfig()
drive = DriveClient.get()

# Baixar cookies se disponível
drive.download_se_ausente(cfg.ID_PASTA_COOKIES, cfg.NOME_COOKIES, Path(cfg.NOME_COOKIES))
cookies_flag = ['--cookies', cfg.NOME_COOKIES] if Path(cfg.NOME_COOKIES).exists() else []

print("═" * 60)
print("📺 BAIXANDO LEGENDAS DO YOUTUBE")
print("═" * 60)

# ── COLOQUE AS URLs DOS VÍDEOS COM LEGENDAS ABAIXO ────────────────────────────
URLS = {
    'pt': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',  # ← IMPORTANTE! Coloque a URL do vídeo em português
    'en': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'es': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'fr': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
}

for lang, url in URLS.items():
    if 'SEU_VIDEO' in url or 'COLOQUE_URL' in url or 'AQUI' in url:
        print(f'\n⚠️  {lang.upper()}: URL não configurada — pulando')
        print(f'   Configure a URL no dicionário URLS acima')
        continue

    nome_srt = cfg.nome_srt(lang)
    print(f'\n⬇️  Baixando {lang.upper()} de: {url[:50]}...')

    cmd = [
        'yt-dlp', '--write-sub', '--sub-lang', lang, '--write-auto-sub',
        '--skip-download', '--sub-format', 'srt', '--convert-subs', 'srt',
        '--output', f'{cfg.NOME_ORACAO}_{lang}', *cookies_flag, url
    ]

    resultado = subprocess.run(cmd, capture_output=True, text=True)

    # Procura o arquivo baixado
    encontrado = False
    for c in Path('.').glob(f'{cfg.NOME_ORACAO}_{lang}*.srt'):
        if c.name != nome_srt:
            c.rename(nome_srt)
        print(f'   ✅ {nome_srt} salvo')
        encontrado = True
        break

    if not encontrado:
        print(f'   ⚠️  Legenda {lang.upper()} não disponível nesse vídeo')
        if resultado.stderr:
            print(f'   Detalhe: {resultado.stderr[:200]}')

# ═══════════════════════════════════════════════════════════════════
# 🆕 CRIAR REFERÊNCIA PT PARA O GROQ (usado na FASE 3)
# ═══════════════════════════════════════════════════════════════════

print("\n" + "═" * 60)
print("📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)")
print("═" * 60)

srt_pt = Path(cfg.nome_srt('pt'))
if srt_pt.exists():
    # Cria o arquivo com o nome que o groq_client espera
    ref_path = Path(f'{cfg.NOME_ORACAO}_yt_ref_pt.srt')
    shutil.copy(srt_pt, ref_path)

    # Mostra preview da referência
    from srt_utils import ler_srt
    ref_legendas = ler_srt(ref_path)
    print(f'✅ Referência PT salva: {ref_path}')
    print(f'   📊 {len(ref_legendas)} segmentos carregados')
    print(f'   📝 Preview: {ref_legendas[0].texto[:80]}...')
    print(f'\n💡 O Groq usará esta referência para corrigir o texto do Whisper na FASE 3')
else:
    print(f'⚠️  Nenhuma legenda PT encontrada em {srt_pt}')
    print(f'   Configure a URL do português no dicionário URLS acima')
    print(f'   O Groq corrigirá sem referência (resultado pode ser menos preciso)')

print("\n" + "═" * 60)
print("✅ B0 CONCLUÍDO")
print("═" * 60)

# Listar arquivos baixados
print("\n📁 Arquivos baixados:")
for srt_file in Path('.').glob(f'{cfg.NOME_ORACAO}*.srt'):
    tamanho = srt_file.stat().st_size / 1024
    print(f"   📄 {srt_file.name} ({tamanho:.1f} KB)")

════════════════════════════════════════════════════════════
📺 BAIXANDO LEGENDAS DO YOUTUBE
════════════════════════════════════════════════════════════

⬇️  Baixando PT de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_pt.srt salvo

⬇️  Baixando EN de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_en.srt salvo

⬇️  Baixando ES de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_es.srt salvo

⬇️  Baixando FR de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_fr.srt salvo

════════════════════════════════════════════════════════════
📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)
════════════════════════════════════════════════════════════
✅ Referência PT salva: pai_nosso_yt_ref_pt.srt
   📊 11 segmentos carregados
   📝 Preview: Pai nosso que estais no  céu,...

💡 O Groq usará esta referência para corrigir o texto do Whisper na FASE 3

════════════════════════════════════════════════════════════
✅ B0 CONCLUÍDO
══════════════════

---
### ⚙️ Inicialização — rode após o Setup

In [16]:
# ╔══════════════════════════════════════════════════╗
# ║  INICIALIZAÇÃO                                 ║
# ╚══════════════════════════════════════════════════╝
from config import PipelineConfig
from groq_client import GroqClient
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint
from google.colab import userdata

# ── Ajuste aqui para trocar a oração ─────────────────────────────────────────
config = PipelineConfig(
    NOME_ORACAO  = 'pai_nosso',
    # Para outra oração:
    # NOME_ORACAO  = 'ave_maria',
    # TEXTO_ORACAO = 'Ave Maria, cheia de graça...',
    # VOZ_EDGE     = 'pt-BR-FranciscaNeural',
)

groq     = GroqClient(api_key=userdata.get('GROQ_KEY'))
pipeline = VideoPipeline(config, groq)

# Estado atual
cp = Checkpoint()
print(config.resumo())
print()
print(cp.resumo())
print(f'\n▶  Próxima fase: {cp.proxima_fase_pendente() or "(tudo concluído)"}')

Oração:        pai_nosso
Áudio:         pai_nosso_audio.wav
SRT mestre:    pai_nosso_pt.srt
Vídeo final:   pai_nosso_final.mp4
Idiomas:       pt, en, es, fr
Voz Edge TTS:  pt-BR-AntonioNeural
Modelo Groq:   llama-3.3-70b-versatile
Duração clipe: 5s
Correcoes:     /content/drive/MyDrive/pipeline/correcoes/pai_nosso
Brutos:        /content/drive/MyDrive/pipeline/brutos/pai_nosso

── Checkpoint ──────────────────────────────
  audio_gerado                 ⬜ pendente
  srt_pt_bruto                 ⬜ pendente
  srt_pt_corrigido             ⬜ pendente
  srt_traduzidos               ⬜ pendente
  classificacoes_feitas        ⬜ pendente
  clipes_cortados              ⬜ pendente
  video_base_criado            ⬜ pendente
  legendas_queimadas           ⬜ pendente
────────────────────────────────────────────

▶  Próxima fase: audio_gerado


In [17]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 INICIALIZAÇÃO DO PIPELINE (com correção do event loop)      ║
# ║  Execute esta célula ANTES da FASE 1                            ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

# PASSO 1: Configurar path para encontrar os módulos
pipeline_path = Path('/content/drive/MyDrive/pipeline')
if str(pipeline_path) not in sys.path:
    sys.path.insert(0, str(pipeline_path))
    print(f"✅ Path configurado: {pipeline_path}")

# PASSO 2: Instalar e aplicar nest_asyncio (resolve o erro do event loop)
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio aplicado - event loop corrigido!")

# PASSO 3: Importar módulos
from config import PipelineConfig
from groq_client import GroqClient
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint
from google.colab import userdata

# PASSO 4: Configurar oração
config = PipelineConfig(
    NOME_ORACAO='pai_nosso',
)

# PASSO 5: Criar cliente Groq
groq = GroqClient(
    api_key=userdata.get('GROQ_KEY'),
    nome_oracao=config.NOME_ORACAO
)

# PASSO 6: Criar pipeline
pipeline = VideoPipeline(config, groq)

# PASSO 7: Verificar status
cp = Checkpoint()
print("\n" + "="*60)
print("✅ PIPELINE INICIALIZADO COM SUCESSO!")
print("="*60)
print(f"📖 Oração: {config.NOME_ORACAO}")
print(f"📁 Checkpoint: {cp.resumo()}")
print(f"▶️  Próxima fase: {cp.proxima_fase_pendente() or '(tudo concluído)'}")
print("="*60)

✅ nest_asyncio aplicado - event loop corrigido!

✅ PIPELINE INICIALIZADO COM SUCESSO!
📖 Oração: pai_nosso
📁 Checkpoint: ── Checkpoint ──────────────────────────────
  audio_gerado                 ⬜ pendente
  srt_pt_bruto                 ⬜ pendente
  srt_pt_corrigido             ⬜ pendente
  srt_traduzidos               ⬜ pendente
  classificacoes_feitas        ⬜ pendente
  clipes_cortados              ⬜ pendente
  video_base_criado            ⬜ pendente
  legendas_queimadas           ⬜ pendente
────────────────────────────────────────────
▶️  Próxima fase: audio_gerado


---
### ▶ Fases 1–4 — Automáticas

In [18]:
# ── FASE 1: Áudio com Edge TTS ───────────────────────────────────────────────
audio = pipeline.fase1_gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')

✅ pai_nosso_audio.wav  (162 KB)


In [19]:
# ── FASE 2: Transcrição Whisper ──────────────────────────────────────────────
srt_bruto = pipeline.fase2_transcrever_whisper()

from srt_utils import ler_srt
print(f'{srt_bruto.name}:')
for leg in ler_srt(srt_bruto)[:4]:
    print(f'  [{leg.inicio_str}]  {leg.texto}')

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


pai_nosso_pt_edge.srt:
  [00:00:00,000]  Pai nosso que estáis no céu, santificados sejam o vosso nome.
  [00:00:05,540]  Vem a nós o vosso reino.
  [00:00:07,800]  Seja feita a vossa vontade, assim na Terra como no céu.
  [00:00:12,940]  O pão nosso de cada dia nos dá hoje.


In [20]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 3 — Correção PT com Groq (com delay, retry e referência) ║
# ║  (usa automaticamente o arquivo *_yt_ref_pt.srt se existir)    ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import time
import re
from datetime import datetime

# PASSO 1: Verificar se existe referência PT do YouTube
ref_path = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')
print("═" * 60)
print("📺 VERIFICANDO REFERÊNCIA PARA CORREÇÃO")
print("═" * 60)

if ref_path.exists():
    from srt_utils import ler_srt
    ref_legendas = ler_srt(ref_path)
    print(f"✅ REFERÊNCIA ENCONTRADA: {ref_path.name}")
    print(f"   📊 {len(ref_legendas)} segmentos")
    print(f"   📝 Primeira linha: {ref_legendas[0].texto[:80]}...")
    print(f"\n💡 O Groq usará esta referência para corrigir o texto do Whisper")
else:
    print(f"⚠️  REFERÊNCIA NÃO ENCONTRADA: {ref_path.name}")
    print(f"   Execute a célula B0 primeiro para baixar a legenda PT do YouTube")
    print(f"   Ou prossiga sem referência (correção pode ser menos precisa)")

print("\n" + "═" * 60)
print("🚀 INICIANDO CORREÇÃO DO TEXTO PT")
print("═" * 60)

# PASSO 2: Função para corrigir com retry automático (em caso de rate limit)
def corrigir_com_retry(max_tentativas=5):
    for tentativa in range(1, max_tentativas + 1):
        try:
            print(f"\n🔄 Tentativa {tentativa}/{max_tentativas} - {datetime.now().strftime('%H:%M:%S')}")
            resultado = pipeline.fase3_corrigir_pt()
            return resultado

        except Exception as e:
            erro_msg = str(e)

            # Verifica se é rate limit (429)
            if '429' in erro_msg or 'rate limit' in erro_msg.lower():
                # Tenta extrair o tempo de espera da mensagem de erro
                match = re.search(r'(\d+)m(\d+\.?\d*)s', erro_msg)
                if match:
                    minutos = int(match.group(1))
                    segundos = float(match.group(2))
                    espera = minutos * 60 + segundos
                else:
                    espera = 30  # espera padrão de 30 segundos

                print(f"\n⏳ RATE LIMIT atingido!")
                print(f"   Aguardando {espera:.0f} segundos antes de tentar novamente...")

                # Barra de progresso
                for i in range(int(espera), 0, -5):
                    print(f"   ⏰ {i} segundos restantes...", end='\r')
                    time.sleep(5)

                print(f"\n   Reiniciando tentativa {tentativa + 1}...\n")
                continue
            else:
                # Outro erro diferente de rate limit
                raise

    raise Exception(f"Falha após {max_tentativas} tentativas devido a rate limit")

# PASSO 3: Executar a correção
try:
    legendas_pt = corrigir_com_retry(max_tentativas=5)

    print("\n" + "═" * 60)
    print(f"✅ CORREÇÃO CONCLUÍDA! {len(legendas_pt)} legendas PT corrigidas")
    print("═" * 60)
    print("\n📝 LEGENDAS CORRIGIDAS:")
    print("─" * 50)
    for leg in legendas_pt:
        print(f'  #{leg.id:02d}  {leg.texto}')
    print("─" * 50)

except Exception as e:
    print(f'\n❌ Erro na correção: {e}')

════════════════════════════════════════════════════════════
📺 VERIFICANDO REFERÊNCIA PARA CORREÇÃO
════════════════════════════════════════════════════════════
✅ REFERÊNCIA ENCONTRADA: pai_nosso_yt_ref_pt.srt
   📊 11 segmentos
   📝 Primeira linha: Pai nosso que estais no  céu,...

💡 O Groq usará esta referência para corrigir o texto do Whisper

════════════════════════════════════════════════════════════
🚀 INICIANDO CORREÇÃO DO TEXTO PT
════════════════════════════════════════════════════════════

🔄 Tentativa 1/5 - 10:24:19
   📺 Referência carregada: pai_nosso_yt_ref_pt.srt (11 segmentos)
   📺 Texto de referência: Pai nosso que estais no  céu, santificado seja o vosso nome, venha a nós o vosso reino, seja feita a vossa vontade, assim na terra como no céu. O pão nosso de cada dia nos dai hoje. Perdoai as nossas ...
   Corrigindo legenda 1/7: 'Pai nosso que estáis no céu, santificados sejam o ...'
   Corrigindo legenda 2/7: 'Vem a nós o vosso reino....'
   Corrigindo legenda 3/7: 'Seja 

In [ ]:
# ── FASE 4: Traduções EN/ES/FR ───────────────────────────────────────────────
from srt_utils import ler_srt
if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)

pipeline.legendas_idiomas = pipeline.fase4_traduzir(legendas_pt)

print('Primeiras 2 legendas por idioma:')
for i in range(min(2, len(legendas_pt))):
    for lang in config.IDIOMAS:
        leg = pipeline.legendas_idiomas.get(lang, [])[i] if i < len(pipeline.legendas_idiomas.get(lang, [])) else None
        if leg:
            print(f'  {lang.upper()}: {leg.texto}')
    print()

---
### ▶ Fase 5A — Classificação morfológica (Groq)

O pipeline vai:
1. Verificar quais idiomas já têm JSONs corrigidos no Drive
2. Classificar os que faltam via Groq
3. Exportar o pacote de revisão (JSONs + CSV + prompt)
4. **Pausar** e mostrar instruções


In [21]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5A — Classificação Groq (FORÇA classificação do zero)    ║
# ║  IGNORA qualquer JSON existente no Drive                        ║
# ╚══════════════════════════════════════════════════════════════════╝

from srt_utils import ler_srt
from video_pipeline import ClassificacaoPendenteError
import time
import re
from datetime import datetime

# Carregar legendas se necessário
if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)
if not pipeline.legendas_idiomas:
    pipeline.legendas_idiomas = pipeline._carregar_todos_srts(legendas_pt)

print("═" * 65)
print("🤖 CLASSIFICAÇÃO MORFOLÓGICA COM GROQ")
print("═" * 65)
print("\n⚠️  Ignorando JSONs existentes no Drive")
print("   Forçando classificação do zero via Groq...")
print("═" * 65)

# Função para classificar com retry automático (em caso de rate limit)
def classificar_com_retry(max_tentativas=5):
    for tentativa in range(1, max_tentativas + 1):
        try:
            print(f"\n🚀 Tentativa {tentativa}/{max_tentativas} - {datetime.now().strftime('%H:%M:%S')}")

            # Forçar classificação para TODOS os idiomas, ignorando cache
            for lang in config.IDIOMAS:
                print(f"\n🔤 Classificando {lang.upper()}...")

                # Limpar palavras existentes para forçar nova classificação
                for leg in pipeline.legendas_idiomas[lang]:
                    leg.palavras = []

                # Forçar classificação via Groq
                pipeline._clf.classificar_idioma(pipeline.legendas_idiomas[lang], lang, forcar=True)
                print(f"   ✅ {lang.upper()} classificado")

                # Delay entre idiomas para evitar rate limit
                if lang != config.IDIOMAS[-1]:
                    print(f"   ⏳ Aguardando 5 segundos...")
                    time.sleep(5)

            # Salvar checkpoint
            pipeline._cp.salvar("classificacoes_feitas", {"fonte": "groq_forcado"})
            return pipeline.legendas_idiomas

        except Exception as e:
            erro_msg = str(e)

            # Verifica se é rate limit (429)
            if '429' in erro_msg or 'rate limit' in erro_msg.lower():
                match = re.search(r'(\d+)m(\d+\.?\d*)s', erro_msg)
                if match:
                    minutos = int(match.group(1))
                    segundos = float(match.group(2))
                    espera = minutos * 60 + segundos
                else:
                    espera = 30

                print(f"\n⏳ RATE LIMIT atingido!")
                print(f"   Aguardando {espera:.0f} segundos antes de tentar novamente...")

                # Barra de progresso
                for i in range(int(espera), 0, -5):
                    print(f"   ⏰ {i} segundos restantes...", end='\r')
                    time.sleep(5)

                print(f"\n   Reiniciando tentativa {tentativa + 1}...\n")
                continue
            else:
                raise

    raise Exception(f"Falha após {max_tentativas} tentativas devido a rate limit")

# Executar classificação forçada
try:
    pipeline.legendas_idiomas = classificar_com_retry(max_tentativas=5)

    print("\n" + "═" * 65)
    print("✅ CLASSIFICAÇÃO CONCLUÍDA!")
    print("═" * 65)
    print("\n📦 Os JSONs brutos foram salvos em:")
    print("   /content/classificacao_pai_nosso_*.json")
    print("\n📁 Backup automático salvo em:")
    print("   MyDrive/pipeline/brutos/pai_nosso/")
    print("\n📋 PRÓXIMOS PASSOS:")
    print("   1. Execute a célula 'GERAR PACOTE DE REVISÃO'")
    print("   2. Corrija os JSONs com a IA")
    print("   3. Salve os JSONs corrigidos no Drive")
    print("   4. Execute a FASE 5C")
    print("═" * 65)

except Exception as e:
    print(f'\n❌ Erro: {e}')

═════════════════════════════════════════════════════════════════
🤖 CLASSIFICAÇÃO MORFOLÓGICA COM GROQ
═════════════════════════════════════════════════════════════════

⚠️  Ignorando JSONs existentes no Drive
   Forçando classificação do zero via Groq...
═════════════════════════════════════════════════════════════════

🚀 Tentativa 1/5 - 10:26:50

🔤 Classificando PT...
   ✅ PT classificado
   ⏳ Aguardando 5 segundos...

🔤 Classificando EN...
   ✅ EN classificado
   ⏳ Aguardando 5 segundos...

🔤 Classificando ES...
   ✅ ES classificado
   ⏳ Aguardando 5 segundos...

🔤 Classificando FR...
   ✅ FR classificado

═════════════════════════════════════════════════════════════════
✅ CLASSIFICAÇÃO CONCLUÍDA!
═════════════════════════════════════════════════════════════════

📦 Os JSONs brutos foram salvos em:
   /content/classificacao_pai_nosso_*.json

📁 Backup automático salvo em:
   MyDrive/pipeline/brutos/pai_nosso/

📋 PRÓXIMOS PASSOS:
   1. Execute a célula 'GERAR PACOTE DE REVISÃO'
   2. C

In [23]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 F 5B GERAR PACOTE DE REVISÃO (usando estrutura correta do Drive) ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import json
import csv
import shutil
from datetime import datetime
from constants import CORES_HTML, IDIOMAS

NOME_ORACAO = config.NOME_ORACAO

# Pastas do Drive
pasta_brutos = Path(f'/content/drive/MyDrive/pipeline/brutos/{NOME_ORACAO}')
pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{NOME_ORACAO}')
pasta_correcoes.mkdir(parents=True, exist_ok=True)

print("═" * 65)
print("📦 GERANDO PACOTE DE REVISÃO")
print("═" * 65)

# 1. COPIAR JSONs BRUTOS para a pasta de CORREÇÕES
print("\n📋 Copiando JSONs brutos para pasta de correções...")
for lang in IDIOMAS:
    json_bruto = pasta_brutos / f'classificacao_{NOME_ORACAO}_{lang}.json'
    json_correcao = pasta_correcoes / f'classificacao_{NOME_ORACAO}_{lang}.json'

    if json_bruto.exists():
        shutil.copy(json_bruto, json_correcao)
        print(f"   ✅ {lang.upper()} - copiado de brutos/ para correcoes/")
    else:
        print(f"   ❌ {lang.upper()} - não encontrado em brutos/")

print("\n✅ JSONs copiados!")
print(f"   📁 De: {pasta_brutos}")
print(f"   📁 Para: {pasta_correcoes}")
print("\n📝 Agora você pode corrigir os JSONs na pasta 'correcoes/'")
print("   Depois de corrigir, execute a FASE 5C")

═════════════════════════════════════════════════════════════════
📦 GERANDO PACOTE DE REVISÃO
═════════════════════════════════════════════════════════════════

📋 Copiando JSONs brutos para pasta de correções...
   ✅ PT - copiado de brutos/ para correcoes/
   ✅ EN - copiado de brutos/ para correcoes/
   ✅ ES - copiado de brutos/ para correcoes/
   ✅ FR - copiado de brutos/ para correcoes/

✅ JSONs copiados!
   📁 De: /content/drive/MyDrive/pipeline/brutos/pai_nosso
   📁 Para: /content/drive/MyDrive/pipeline/correcoes/pai_nosso

📝 Agora você pode corrigir os JSONs na pasta 'correcoes/'
   Depois de corrigir, execute a FASE 5C


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5C — RECARREGAR CLASSIFICAÇÕES CORRIGIDAS DO DRIVE       ║
# ║  (execute DEPOIS de colocar os JSONs corrigidos no Drive)      ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 65)
print("📥 RECARREGANDO CLASSIFICAÇÕES CORRIGIDAS DO DRIVE")
print("═" * 65)

# Verificar se os JSONs corrigidos existem no Drive
pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{config.NOME_ORACAO}')

if not pasta_correcoes.exists():
    print(f"❌ Pasta não encontrada: {pasta_correcoes}")
    print("   Execute a célula 'GERAR PACOTE DE REVISÃO' primeiro")
else:
    print(f"📁 Verificando pasta: {pasta_correcoes}")

    # Listar JSONs encontrados
    jsons_encontrados = []
    for lang in config.IDIOMAS:
        json_path = pasta_correcoes / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
        if json_path.exists():
            jsons_encontrados.append(lang.upper())
            print(f"   ✅ {json_path.name}")
        else:
            print(f"   ❌ classificacao_{config.NOME_ORACAO}_{lang}.json")

    if len(jsons_encontrados) == 4:
        print("\n✅ Todos os 4 JSONs encontrados! Carregando...")

        # Recarregar classificações
        pipeline.legendas_idiomas = pipeline.fase5b_recarregar()

        print("\n" + "═" * 65)
        print("✅ CLASSIFICAÇÕES CARREGADAS COM SUCESSO!")
        print("═" * 65)
        print("\n🎬 Agora você pode executar as FASES 6, 7 e 8")

    else:
        print(f"\n⚠️ Faltam {4 - len(jsons_encontrados)} JSON(s)")
        print("   Coloque os JSONs corrigidos na pasta e execute novamente")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 GERAR PACOTE DE REVISÃO PARA IA                             ║
# ║  (execute após a FASE 5A, antes de corrigir com a IA)          ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 65)
print("📦 GERANDO PACOTE DE REVISÃO")
print("═" * 65)

# Exportar pacote de revisão
pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)

print(f"\n✅ Pacote gerado em: {pacote}")
print("\n📁 Arquivos criados:")
print("   • prompt_revisao.md")
print("   • relatorio_classificacoes.csv")
print("   • classificacao_pai_nosso_pt.json")
print("   • classificacao_pai_nosso_en.json")
print("   • classificacao_pai_nosso_es.json")
print("   • classificacao_pai_nosso_fr.json")

print("\n" + "═" * 65)
print("📋 PRÓXIMOS PASSOS:")
print("═" * 65)
print("1. Acesse: MyDrive/pipeline/correcoes/pai_nosso/")
print("2. Copie o conteúdo do prompt_revisao.md")
print("3. Cole na IA junto com os 4 JSONs")
print("4. A IA corrige e você salva os JSONs na mesma pasta")
print("5. Execute a FASE 5C")
print("═" * 65)

---
### ⏸️ Fase 5B — Revisão manual pela IA ← **Você faz isso**

Após a Fase 5A pausar:

1. **Abra o Google Drive** → `MyDrive/pipeline/correcoes/pai_nosso/`
2. Você verá os arquivos:
   - `classificacao_pai_nosso_pt.json` (e en, es, fr)
   - `relatorio_classificacoes.csv`
   - `prompt_revisao.md` ← **copie e cole numa IA (Claude, GPT, etc.)**
3. Anexe os 4 JSONs junto com o prompt na conversa da IA
4. A IA vai retornar os JSONs corrigidos
5. **Salve os JSONs corrigidos na mesma pasta do Drive** (substituindo os originais)
6. Execute a célula abaixo (Fase 5C)


---
### ▶ Fase 5C — Recarregar classificações corrigidas

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  FASE 5C — Recarregar do Drive após revisão   ║
# ║  Execute APÓS colocar os JSONs corrigidos      ║
# ║  na pasta Drive.                              ║
# ╚══════════════════════════════════════════════════╝

pipeline.legendas_idiomas = pipeline.fase5b_recarregar()

# Preview: mostra a primeira legenda de cada idioma com suas classes
print('\n📋 Preview das classificações carregadas:')
print('─' * 55)
for lang in config.IDIOMAS:
    legendas = pipeline.legendas_idiomas.get(lang, [])
    if legendas and legendas[0].palavras:
        leg = legendas[0]
        print(f'\n  {lang.upper()} — "{leg.texto[:50]}"')
        for p in leg.palavras[:5]:
            from constants import CORES_HTML
            cor = CORES_HTML.get(p.classe, '❌ SEM COR')
            valida = '✅' if p.classe in CORES_HTML else '❌'
            print(f'    {valida} {p.texto:<18} {p.classe:<35} {cor}')
        if len(leg.palavras) > 5:
            print(f'       ... (+{len(leg.palavras)-5} palavras)')

---
### ▶ Fases 6–8 — Vídeo final (rode após a Fase 5C)

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  FASES 6, 7 e 8 — Video final                ║
# ╚══════════════════════════════════════════════════╝
# Roda automaticamente após as classificações estarem prontas.

video_final = pipeline.continuar()

if video_final:
    print(f'\n🎬 Vídeo final: {video_final}')
    print(f'   Tamanho: {video_final.stat().st_size / 1_048_576:.1f} MB')
    print(f'   Salvo no Drive: pasta Vídeos')

---
### 🚀 RUN ALL — Pipeline completo com checkpoint

Use quando quiser rodar tudo de uma vez. O pipeline pausa automaticamente na Fase 5A
se houver classificações a revisar. Após a revisão, rode `pipeline.fase5b_recarregar()`
e `pipeline.continuar()`.


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  RUN ALL                                       ║
# ╚══════════════════════════════════════════════════╝
from video_pipeline import ClassificacaoPendenteError

resultado = pipeline.run(
    # from_phase='clipes_cortados'  # descomente para retomar de uma fase
)

if resultado is None:
    print('\n⏸️  Pipeline pausado na Fase 5A.')
    print('   Siga as instruções acima, depois execute a célula FASE 5C.')
else:
    print(f'\n🎉 Vídeo final: {resultado}')
    print(pipeline._cp.resumo())

---
### 🔧 Utilitários

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  UTILITÁRIOS — descomente o que precisar       ║
# ╚══════════════════════════════════════════════════╝
from checkpoint import Checkpoint
from pathlib import Path

cp = Checkpoint()
print(cp.resumo())
print()
pipeline._clf.imprimir_status()

# ── Checkpoint ────────────────────────────────────────────────────────────────

# Resetar tudo (começar do zero):
# cp.resetar_tudo()

# Reiniciar a partir de uma fase:
# cp.reiniciar_de('classificacoes_feitas')

# ── Classificações ────────────────────────────────────────────────────────────

# Ver status detalhado das classificações:
# pipeline._clf.imprimir_status()

# Forçar nova classificação via Groq (ignora cache):
# from classification import Classifier
# pipeline._clf.classificar_idioma(pipeline.legendas_idiomas['en'], 'en', forcar=True)

# Exportar novo pacote de revisão manualmente:
# pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)

# Verificar classes sem cor (problemas no JSON):
# from constants import CORES_HTML
# for lang, legendas in pipeline.legendas_idiomas.items():
#     for leg in legendas:
#         for p in leg.palavras:
#             if p.classe not in CORES_HTML:
#                 print(f'  [{lang.upper()} leg.{leg.id}] "{p.texto}" → {p.classe} ❌')

# ── Vídeo / limpeza ───────────────────────────────────────────────────────────

# Inspecionar ASS gerado:
# ass = Path(f'legendas_{config.NOME_ORACAO}.ass')
# if ass.exists():
#     print(ass.read_text()[:3000])

# Limpar temporários (clipes, vídeos intermediários):
# pipeline.limpar_temporarios()